(mmm_control_dimensionality)=
# Control Dimensionality and ROAS

How many control variables should an MMM include, and what does the answer cost you?

The instinct is that controls are cheap. They are not the treatment, they are not what you report, and leaving one out risks omitted-variable bias — so the temptation is to throw in everything: holidays, weather, competitor pricing, macro indices, one dummy per promotion. This notebook shows that with the default prior on control coefficients, that instinct quietly damages the number you actually care about: **total media ROAS**.

The problem is not confounding. We deliberately generate controls that are *independent of media spend*, so they cannot bias ROAS through a back-door path — in the vocabulary of [Cinelli, Forney and Pearl (2024)](https://doi.org/10.1177/00491241221099552) they are **neutral controls**. Everything we observe is therefore attributable to prior geometry alone.

## What goes wrong, in one sentence

PyMC-Marketing gives each control coefficient an independent Normal prior whose scale does not depend on how many controls there are. Add controls and the *total* variance the model expects the control block to explain grows linearly in $K$, while the prior on the residual scale stays put. The implied prior on $R^2$ therefore marches towards 1, and by the time $K$ is comparable to the number of observations the control block is free enough to absorb — or manufacture — variation that belongs to media.

The fix is a prior that budgets a *fixed* amount of explained variance and splits it across however many controls happen to be present. That is the **R2D2** prior of [Zhang, Naughton, Bondell and Reich (2022)](https://doi.org/10.1080/01621459.2020.1825449): put a prior on $R^2$, convert it into a total coefficient-variance budget, and divide that budget over the coefficients with a Dirichlet.

## Relationship to the source paper

This notebook is a media-mix translation of Experiment 4 of [*To select or not to select*](https://arxiv.org/abs/2606.22850) (Section 5.4). The paper studies a randomised experiment with a treatment $z$, a treatment effect $\alpha$, and $p$ neutral covariates, and asks how the posterior for $\alpha$ evolves as $p$ grows under different priors. The mapping:

| Paper | This notebook |
| --- | --- |
| treatment $z$ | media spend, passed through adstock and saturation |
| treatment effect $\alpha$ | **total media ROAS** |
| covariates $X$, drawn iid, independent of $z$ | control variables, drawn iid, independent of spend |
| $M_{\text{base}}$ (treatment only) | an MMM with `control_columns=None` |
| $M_{\text{full}}$ at $p \in \{10, 50, 100\}$ | an MMM fit on the first $K$ controls |
| $n_{\text{obs}} = 150$ | $n = 156$ weeks (three years of weekly data) |

Two details of the mapping are worth flagging up front. First, the base model is the *punchline*, not a footnote: Table 4 of the paper reports that the Normal full model is **worse** than the base model (RMSE 0.30 vs 0.27, coverage 0.87 vs 0.92), so "add the controls, they cannot hurt" is exactly the belief under test. Second, of the paper's three prior specifications we implement **Normal** and **split** (R2D2 on the controls, media priors untouched). The joint R2D2, which shrinks the media coefficients too and which the paper shows to be badly biased, is out of scope.

We also go one step beyond the paper's grid. The paper stops at $p/n \approx 0.67$; we add $K = 200 > n$, because an MMM with three years of weekly data and a few hundred candidate control columns is an entirely ordinary situation, and it is where the two priors part company most clearly.

## Prepare Notebook

In [ ]:
import warnings

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pymc.dims as pmd
import pytensor.tensor as pt
import seaborn as sns
import xarray as xr
from pymc_extras.prior import Prior
from pytensor.xtensor.type import as_xtensor

from pymc_marketing.mmm import GeometricAdstock, LogisticSaturation
from pymc_marketing.mmm.mmm import MMM
from pymc_marketing.mmm.transformers import geometric_adstock, logistic_saturation
from pymc_marketing.special_priors import SpecialPrior

warnings.filterwarnings("ignore", category=FutureWarning)

az.style.use("arviz-darkgrid")
plt.rcParams["figure.figsize"] = [10, 6]
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.facecolor"] = "white"

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

The experiment is driven by a handful of constants. `N_DATES` and `K_MAX` set the shape of the problem; `K_GRID` is the sequence of nested control subsets we fit.

In [ ]:
SEED = 42

N_DATES = 156  # three years of weekly data
N_CHANNELS = 5
L_MAX = 8  # adstock window, in weeks
K_MAX = 200  # size of the pool of candidate controls
CHOL_ETA = 3.0  # LKJ concentration for the spend correlation matrix

K_GRID = [0, 5, 10, 25, 50, 100, 200]

CHANNELS = [f"x{i + 1}" for i in range(N_CHANNELS)]
CONTROLS = [f"c{k + 1}" for k in range(K_MAX)]
DATES = pd.date_range("2021-01-04", freq="W-MON", periods=N_DATES)

## The data-generating process

We need a dataset where the true total ROAS is known exactly and identical for every $K$, so that any movement in the posterior is attributable to the control prior and nothing else. We build it in three pieces: spend, controls, and the target.

### Media spend

Channel spend comes from an LKJ-Cholesky generative model, the same construction used in the {ref}`parameter recovery notebook <mmm_data_generator>`. It gives realistically correlated channel spend with per-channel trends, and `softplus` enforces non-negativity.

One deviation from that notebook: the per-channel scale prior is `Gamma(3, 1)` rather than `Exponential(1/3)`. Both have mean 3, but the Exponential regularly draws values near zero, which produces a channel whose spend barely moves across the three years. Because the MMM divides each channel by its own maximum before saturating, such a channel sits permanently at the flat top of its saturation curve and contributes an almost constant term — one that is not separable from the intercept. The Gamma keeps every channel's spend on a usable dynamic range.

In [ ]:
def draw_spend(rng: np.random.Generator) -> np.ndarray:
    """Draw one realisation of correlated, trending, non-negative channel spend."""
    coords = {"channel": CHANNELS, "date": np.arange(N_DATES)}
    t = np.arange(N_DATES) / N_DATES

    with pm.Model(coords=coords) as spend_model:
        t_data = pm.Data("t", t, dims=("date",))
        L, _, _ = pm.LKJCholeskyCov(
            "L",
            n=N_CHANNELS,
            eta=CHOL_ETA,
            sd_dist=pm.Gamma.dist(alpha=3, beta=1),
        )
        a = pm.Normal("a", mu=0, sigma=1, dims="channel")
        b = pm.Normal("b", mu=0, sigma=1, dims="channel")
        mu = pm.Deterministic("mu", a + b * t_data[..., None], dims=("date", "channel"))
        x_raw = pm.MvNormal("x_raw", mu=mu, chol=L, dims=("date", "channel"))
        pm.Deterministic("x", pt.softplus(x_raw), dims=("date", "channel"))

    return pm.draw(spend_model.x, draws=1, random_seed=rng)

### The true media contribution

We call `geometric_adstock` and `logistic_saturation` from `pymc_marketing.mmm.transformers` directly, at fixed true `alpha` and `lam`, so the truth matches the model's functional form *exactly* rather than approximately. Two settings have to line up with what the `MMM` class does internally, or the truth will silently disagree with the model:

- the transformations are applied to **max-scaled** spend, `x / x.max()`, because `MMM` max-scales each channel before the forward pass;
- `normalize=True`, because that is the default on the `GeometricAdstock` wrapper even though the underlying `geometric_adstock` function defaults to `False`.

`beta` enters `LogisticSaturation` as a pure multiplicative scale, so evaluating the shape once at `beta = 1` lets us solve the variance budget below in closed form. This is why we call the transformers directly instead of routing the target through `pm.do` and `sample_prior_predictive`, which would need a two-pass loop to hit a variance target.

In [ ]:
def media_shape(
    spend_scaled: np.ndarray, alpha: np.ndarray, lam: np.ndarray
) -> np.ndarray:
    """Adstock then saturation at unit beta, matching the MMM's forward pass."""
    adstocked = geometric_adstock(
        as_xtensor(spend_scaled, dims=("date", "channel")),
        alpha=as_xtensor(alpha, dims=("channel",)),
        l_max=L_MAX,
        dim="date",
        normalize=True,
    )
    saturated = logistic_saturation(adstocked, lam=as_xtensor(lam, dims=("channel",)))
    return saturated.transpose("date", "channel").values.eval()

### Controls, variance budget, and the target

The controls are $K_{\max}$ iid standard normal columns, standardised and independent of spend. Every one of them carries the *same* true coefficient, mirroring the paper's `beta <- array(0.1, c(p, 1))`. Standardising matters here because, unlike the target and the channels, `MMM` does **not** scale controls — so `gamma_control` lives in scaled-target-per-raw-control units, and standardised controls are what keep that interpretable.

The rest of the DGP is a variance budget with three knobs:

- **`media_contribution_share`** — the fraction of total sales driven by media. Since $\mathbb{E}[y] = \text{baseline} + \mathbb{E}[\text{media}]$ and the controls and noise are mean-zero, targeting a share $s$ means $\mathbb{E}[\text{media}] = s \cdot \text{baseline} / (1 - s)$, which fixes the overall scale of `saturation_beta`. Default 0.15.
- **`delta_media`** — media's share of *explained* variance, the analogue of equation (34) in the paper. The media level above already fixes media's variance, so this knob sets the control block's variance. Default 0.25: the controls do most of the explaining, as they usually do in a real MMM.
- **`true_r2`** — the fraction of the target's variance that media and controls jointly explain, which pins the residual scale (the paper's footnote 15). Default 0.9.

Finally we divide the target by its own maximum. This is not cosmetic: `MMM` max-scales the target, so normalising here makes the model's internal scale exactly 1 and lets us report every true parameter in the units the model actually works in.

In [ ]:
def simulate(
    seed: int,
    *,
    media_contribution_share: float = 0.15,
    true_r2: float = 0.9,
    delta_media: float = 0.25,
    baseline: float = 1.0,
    spend_to_revenue: float = 0.075,
) -> tuple[pd.DataFrame, dict]:
    """Generate one dataset together with its exact ground truth."""
    rng = np.random.default_rng(seed)

    spend_raw = draw_spend(rng)
    spend_scaled = spend_raw / spend_raw.max(axis=0)

    alpha_true = rng.beta(1, 3, size=N_CHANNELS)
    lam_true = rng.gamma(shape=3, scale=1, size=N_CHANNELS)
    shape = media_shape(spend_scaled, alpha_true, lam_true)

    # Relative betas that equalise each channel's mean contribution, so that no
    # single channel dominates the media term.
    beta_rel = 1.0 / shape.mean(axis=0)
    shape_unit = shape @ beta_rel

    controls = rng.normal(size=(N_DATES, K_MAX))
    controls = (controls - controls.mean(axis=0)) / controls.std(axis=0)

    media_mean = media_contribution_share * baseline / (1 - media_contribution_share)
    beta_true = beta_rel * media_mean / shape_unit.mean()
    media = shape @ beta_true

    var_media = media.var()
    var_control = var_media * (1 - delta_media) / delta_media
    var_explained = var_media + var_control

    control_sum = controls.sum(axis=1)
    gamma_true = np.sqrt(var_control / control_sum.var())
    control = control_sum * gamma_true

    sigma_true = np.sqrt(var_explained * (1 - true_r2) / true_r2)
    noise = rng.normal(scale=sigma_true, size=N_DATES)

    y_raw = baseline + media + control + noise
    scale = y_raw.max()
    y = y_raw / scale

    spend_level = spend_to_revenue * y.sum() / spend_raw.sum()
    spend = spend_raw * spend_level

    truth = {
        # True parameters, expressed in the model's internal (scaled-target) units.
        "intercept": baseline / scale,
        "adstock_alpha": alpha_true,
        "saturation_lam": lam_true,
        "saturation_beta": beta_true / scale,
        "gamma_control": gamma_true / scale,
        "y_sigma": sigma_true / scale,
        # Realised summaries of this particular dataset.
        "total_roas": (media / scale).sum() / spend.sum(),
        "media_contribution_share": media.sum() / y_raw.sum(),
        "delta_media": var_media / var_explained,
        "r2": (media + control).var() / y_raw.var(),
        "target_scale": scale,
    }

    frame = {"date": DATES}
    frame |= {channel: spend[:, i] for i, channel in enumerate(CHANNELS)}
    frame |= {control_name: controls[:, k] for k, control_name in enumerate(CONTROLS)}
    frame["y"] = y

    return pd.DataFrame(frame), truth

We draw the dataset **once** and hold it fixed for every model below. `y` is therefore byte-identical across all fits, and the nested subsets $K \in$ `K_GRID` all come from the same pool of `K_MAX` columns.

In [ ]:
data, truth = simulate(SEED)

pd.Series(
    {
        "true total ROAS": truth["total_roas"],
        "media share of sales": truth["media_contribution_share"],
        "media share of explained variance": truth["delta_media"],
        "true R-squared": truth["r2"],
        "true residual sigma (scaled target)": truth["y_sigma"],
        "true gamma per control (scaled target)": truth["gamma_control"],
        "target scale": truth["target_scale"],
    }
).round(4)

Ground-truth total ROAS is the total true media contribution divided by total spend — a single number, identical for every $K$ because the data never change.

The controls are independent of spend by construction, but with $n = 156$ the *sample* correlations are not exactly zero. Worth knowing how large they get, since a control that happens to correlate with spend is the mechanism by which a neutral control can still move ROAS.

In [ ]:
spend_matrix = data[CHANNELS].to_numpy()
control_matrix = data[CONTROLS].to_numpy()
cross_corr = np.corrcoef(spend_matrix.T, control_matrix.T)[:N_CHANNELS, N_CHANNELS:]

print(f"max |corr(spend, control)| : {np.abs(cross_corr).max():.3f}")
print(f"mean |corr(spend, control)|: {np.abs(cross_corr).mean():.3f}")

### Looking at the data

In [ ]:
fig, axes = plt.subplots(nrows=3, figsize=(12, 9), sharex=True)

sns.lineplot(x="date", y="y", data=data, color="black", ax=axes[0])
axes[0].set(title="Target (max-scaled sales)", ylabel="y")

for channel in CHANNELS:
    sns.lineplot(x="date", y=channel, data=data, ax=axes[1], label=channel, alpha=0.8)
axes[1].set(title="Channel spend", ylabel="spend")
axes[1].legend(ncol=5, loc="upper left", fontsize=8)

for control_name in CONTROLS[:5]:
    sns.lineplot(
        x="date", y=control_name, data=data, ax=axes[2], alpha=0.6, linewidth=0.9
    )
axes[2].set(
    title=f"First 5 of {K_MAX} controls (iid, standardised, independent of spend)",
    ylabel="control value",
    xlabel="date",
)

fig.suptitle("The fixed dataset, shared by every model below", fontsize=14)
fig.tight_layout()

## The two control priors

Both priors slot into `model_config["gamma_control"]`, which is the whole point: the media priors are untouched **by construction**, which is what makes the second one a *split* prior rather than a joint one. No `MuEffect` subclass and no changes to the `MMM` class are needed.

Here is the code path they feed into. `gamma_control` is created, multiplied by the control data, summed over controls, and added to `mu`:

```python
if self.control_columns is not None and len(self.control_columns) > 0:
    gamma_control = self.model_config["gamma_control"].create_variable(
        name="gamma_control", xdist=True
    )
    control_data_ = pmd.Data("control_data", self.xarray_dataset._control)
    control_contribution = pmd.Deterministic(
        "control_contribution", control_data_ * gamma_control
    )
    mu_var += control_contribution.sum(dim="control")
```

### 1. Normal

The library default is `Prior("Normal", mu=0, sigma=2, dims="control")`. Note what is missing: `sigma` does not depend on $K$. The prior variance the control block expects to explain is $K \sigma^2$, growing without bound as controls are added.

Rather than compare R2D2 against that default — which, as we will see, is already saturated at $K = 5$ — we give the Normal prior every advantage and **calibrate** it. The R2D2 prior below allocates an expected total control-coefficient variance of `SIGMA_REF ** 2`; we choose the Normal `sigma` so that its budget, $K \sigma^2$, matches that at a reference $K$. The result is a Normal prior that is genuinely well tuned at $K = K_{\text{ref}}$ and, crucially, is *still not a function of $K$* — so everything that happens afterwards is a pure dimension effect rather than a badly chosen constant.

`SIGMA_REF` is a plug-in guess at the residual scale. The target is max-scaled into roughly $[0.4, 1]$ and we expect a well-fitting model, so 0.1 is a deliberately loose but not absurd guess; we use the same value for the likelihood's `HalfNormal` scale so that the two priors are stated on a common footing.

In [ ]:
SIGMA_REF = 0.1  # plug-in reference residual scale, on the scaled-target scale
K_REF = 5  # reference dimension at which the Normal prior is calibrated

SIGMA_NORMAL = SIGMA_REF / np.sqrt(K_REF)
print(f"calibrated Normal sigma: {SIGMA_NORMAL:.4f}  (library default is 2)")

### 2. Split R2D2

`gamma_control` is consumed through `create_variable(name="gamma_control", xdist=True)`, and `pymc_marketing.special_priors` already defines a `SpecialPrior` ABC for priors that behave like `Prior` but need custom graph construction — `LaplacePrior` and `LogNormalPrior` are the shipped examples. A new subclass only needs `_checks` and `create_variable`.

The construction, following `r2_normal.stan` from the paper's reference implementation:

$$
R^2 \sim \text{Beta}\!\left(\bar{r}\,\kappa,\ (1-\bar{r})\,\kappa\right), \qquad
\tau^2 = \sigma_{\text{ref}}^2 \frac{R^2}{1 - R^2}, \qquad
\psi \sim \text{Dirichlet}(a \mathbf{1}_K), \qquad
\gamma_k = z_k \sqrt{\tau^2 \psi_k},\quad z_k \sim \mathcal{N}(0, 1).
$$

This is the mechanism that matters: $\tau^2$ is a **fixed** variance budget, and the Dirichlet weights $\psi$ split it among however many controls there are. The total control variance therefore does not grow with $K$. We obtain the Dirichlet through the unnormalised-Gamma trick (draw Gammas, divide by their sum), lifted straight from the reference Stan model, which avoids needing a Dirichlet over a named dim.

Defaults follow the paper's `config.yaml`: `r2_mean = 1/3` and `r2_precision = 3`, giving $\text{Beta}(1, 2)$, with a symmetric Dirichlet concentration of 1.

#### Why not wrap `pymc_extras`?

`pymc_extras.distributions.R2D2M2CP` implements a richer relative of this prior, so the honest first move is to try wrapping it. It does not fit this hook, for three concrete reasons:

1. It returns `(residual_sigma, coefficients)` — it *owns* the residual scale. `create_variable` returns a single variable, and the MMM builds its likelihood `sigma` separately, so there is nowhere to put the first element.
2. It requires `output_sigma` and `input_sigma` as inputs, i.e. the very scales the MMM derives internally.
3. It is built from plain `pm.*` primitives inside a nested submodel, whereas `gamma_control` is consumed as a `pymc.dims` xtensor. It also layers on a correlation-probability (`positive_probs`) component that is not part of the plain R2D2 the paper studies.

The ten lines below are less trouble than bridging those gaps.

In [ ]:
class R2D2Prior(SpecialPrior):
    r"""R2D2 prior: a fixed coefficient-variance budget, split across coefficients.

    An :math:`R^2` prior is converted into a total coefficient variance
    :math:`\tau^2 = \sigma_{\text{ref}}^2 R^2 / (1 - R^2)`, which a Dirichlet
    then splits across the coefficients.  Because the budget is fixed, adding
    coefficients divides the same variance more finely instead of inflating the
    total.

    Parameters
    ----------
    dims : tuple of str
        Dims of the coefficient vector.  The budget is split across the last one.
    r2_mean : float
        Prior mean of :math:`R^2`.
    r2_precision : float
        Precision of the Beta prior on :math:`R^2`; larger is tighter.
    concentration : float
        Symmetric Dirichlet concentration.  1.0 is uniform over the simplex.
    sigma_ref : float
        Plug-in reference residual scale.
    """

    def __init__(
        self,
        dims: tuple | None = None,
        centered: bool = True,
        *,
        r2_mean: float = 1 / 3,
        r2_precision: float = 3.0,
        concentration: float = 1.0,
        sigma_ref: float = 1.0,
    ) -> None:
        super().__init__(
            dims=dims,
            centered=centered,
            r2_mean=r2_mean,
            r2_precision=r2_precision,
            concentration=concentration,
            sigma_ref=sigma_ref,
        )

    def _checks(self) -> None:
        expected = {"r2_mean", "r2_precision", "concentration", "sigma_ref"}
        if set(self.parameters) != expected:
            raise ValueError(f"Parameters must be {sorted(expected)}")
        if not self.dims:
            raise ValueError("R2D2Prior needs at least one dim to split the budget over")

    def create_variable(self, name: str, xdist: bool = False):
        """Build the R2D2 graph and return the coefficient vector."""
        if not xdist:
            raise NotImplementedError(f"{self!r} only supports xdist=True")

        r2_mean = self.parameters["r2_mean"]
        r2_precision = self.parameters["r2_precision"]
        split_dim = self.dims[-1]

        r2 = pmd.Beta(
            f"{name}_r2",
            alpha=r2_mean * r2_precision,
            beta=(1.0 - r2_mean) * r2_precision,
        )
        weights_raw = pmd.Gamma(
            f"{name}_weights_raw",
            alpha=self.parameters["concentration"],
            beta=1.0,
            dims=self.dims,
        )
        weights = weights_raw / weights_raw.sum(split_dim)
        tau2 = self.parameters["sigma_ref"] ** 2 * r2 / (1.0 - r2)
        offset = pmd.Normal(f"{name}_offset", mu=0.0, sigma=1.0, dims=self.dims)

        return pmd.Deterministic(name, offset * pmd.math.sqrt(tau2 * weights))

A quick check that it builds and samples, and that the total variance budget it implies is stable in $K$ — which is the entire claim.

In [ ]:
r2d2_check = R2D2Prior(dims=("control",), sigma_ref=SIGMA_REF)

for k in [5, 50, 200]:
    draws = r2d2_check.sample_prior(
        coords={"control": CONTROLS[:k]},
        name="gamma_control",
        draws=2_000,
        random_seed=SEED,
    )
    budget = (draws["gamma_control"] ** 2).sum("control")
    print(f"K={k:4d}  median total budget sum(gamma^2) = {float(budget.median()):.5f}")

### Two honest caveats

**The reference scale is a plug-in.** `r2_normal.stan` sets `beta = beta_z .* sqrt(sigma^2 * tau2 * psi)`, conditioning the coefficient scale on the *sampled* residual `sigma`. In the MMM, `gamma_control` is created before the likelihood exists, so that variable cannot be referenced; we use a fixed `sigma_ref` instead. This is defensible — the MMM max-scales its target, so the scaled target has $O(1)$ variance and a reasonable guess is available — and it leaves the property the experiment turns on fully intact, since what matters is that the budget is *fixed*, not that it is exactly right. Wiring in the sampled `sigma` would need a lazy hook or an effect that owns the likelihood, and is genuine future work.

**R2D2 assumes standardised covariates** (footnote 7 of the paper). Ours are standardised by construction, which matters more here than in a plain regression because `MMM` does not scale controls itself. With controls on wildly different scales, a single shared budget is not meaningful — standardise them before using this prior.

## Model configuration

Everything except `gamma_control` is held identical across every fit. The priors are stated on the scaled-target scale, where the target lies in roughly $[0.4, 1]$: this is worth being explicit about, because the library defaults (`intercept` and `gamma_control` at `sigma=2`, likelihood `sigma` at `HalfNormal(2)`) are extremely diffuse relative to a max-scaled target, and that diffuseness is a large part of the story.

In [ ]:
BASE_CONFIG = {
    "intercept": Prior("Normal", mu=0.5, sigma=0.5),
    "adstock_alpha": Prior("Beta", alpha=1, beta=3, dims="channel"),
    "saturation_lam": Prior("Gamma", alpha=3, beta=1, dims="channel"),
    "saturation_beta": Prior("HalfNormal", sigma=0.25, dims="channel"),
    "likelihood": Prior(
        "Normal", sigma=Prior("HalfNormal", sigma=SIGMA_REF), dims="date"
    ),
}

CONTROL_PRIORS = {
    "Normal": Prior("Normal", mu=0, sigma=SIGMA_NORMAL, dims="control"),
    "R2D2": R2D2Prior(dims=("control",), sigma_ref=SIGMA_REF),
}

SAMPLER_CONFIG = {
    "draws": 1_000,
    "tune": 1_000,
    "chains": 4,
    "target_accept": 0.99,
    "nuts_sampler": "nutpie",
}

`target_accept=0.99` is applied to *every* fit, Normal and R2D2 alike. The R2D2 prior has a funnel: with $\text{Beta}(1, 2)$ on $R^2$, the budget $\tau^2 = \sigma_{\text{ref}}^2 R^2/(1-R^2)$ has infinite variance, and at small $K$ — where the data barely constrain $R^2$ — the default `target_accept` produces divergences. Raising it removes them. Using the same setting everywhere keeps the comparison clean.

`K = 0` must pass `control_columns=None`, not `[]`: the `MMM` constructor enforces `min_length=1` on that field.

In [ ]:
def build_mmm(k: int, prior_name: str | None = None) -> MMM:
    """Build an MMM on the first `k` controls with the named control prior."""
    config = dict(BASE_CONFIG)
    if k > 0:
        config["gamma_control"] = CONTROL_PRIORS[prior_name]

    return MMM(
        date_column="date",
        channel_columns=CHANNELS,
        control_columns=CONTROLS[:k] if k > 0 else None,
        target_column="y",
        adstock=GeometricAdstock(l_max=L_MAX),
        saturation=LogisticSaturation(),
        model_config=config,
        sampler_config=SAMPLER_CONFIG,
    )

## The mechanism: implied prior $R^2$ against $K$

Before running any MCMC we can see the problem directly, from prior draws alone. This is the analogue of the left panel of Figure 4 in the paper, and it is by far the cheapest diagnostic in this notebook.

For each prior draw we decompose the variance of the linear predictor over time into a media part and a control part, and add the residual variance:

$$
R^2_{\text{media}} = \frac{\operatorname{Var}_t(\text{media})}{\operatorname{Var}_t(\text{media}) + \operatorname{Var}_t(\text{control}) + \sigma^2},
\qquad
R^2_{\text{control}} = \frac{\operatorname{Var}_t(\text{control})}{\cdots},
$$

so the two shares add to the implied prior $R^2$. Everything comes from `sample_prior_predictive`; no sampler is involved.

In [ ]:
def prior_variance_shares(
    k: int, control_prior, draws: int = 2_000
) -> tuple[xr.DataArray, xr.DataArray]:
    """Media and control shares of implied prior variance, per prior draw."""
    config = dict(BASE_CONFIG)
    if k > 0:
        config["gamma_control"] = control_prior

    mmm = MMM(
        date_column="date",
        channel_columns=CHANNELS,
        control_columns=CONTROLS[:k] if k > 0 else None,
        target_column="y",
        adstock=GeometricAdstock(l_max=L_MAX),
        saturation=LogisticSaturation(),
        model_config=config,
    )
    mmm.build_model(data.drop(columns=["y"]), data["y"])

    var_names = ["channel_contribution", "y_sigma"]
    if k > 0:
        var_names.append("control_contribution")

    with mmm.model:
        prior = pm.sample_prior_predictive(
            draws=draws, var_names=var_names, random_seed=SEED
        ).prior.to_dataset()

    var_media = prior["channel_contribution"].sum("channel").var("date")
    var_control = (
        prior["control_contribution"].sum("control").var("date")
        if k > 0
        else xr.zeros_like(var_media)
    )
    total = var_media + var_control + prior["y_sigma"] ** 2

    return var_media / total, var_control / total

In [ ]:
PRIOR_VARIANTS = {
    "Normal, library default (sigma=2)": Prior("Normal", mu=0, sigma=2, dims="control"),
    f"Normal, calibrated (sigma={SIGMA_NORMAL:.3f})": CONTROL_PRIORS["Normal"],
    "Split R2D2": CONTROL_PRIORS["R2D2"],
}

prior_records = []
for label, control_prior in PRIOR_VARIANTS.items():
    for k in K_GRID:
        media, control = prior_variance_shares(k, control_prior)
        r2 = media + control
        prior_records.append(
            {
                "prior": label,
                "K": k,
                "r2_median": float(r2.median()),
                "r2_q05": float(r2.quantile(0.05)),
                "r2_q95": float(r2.quantile(0.95)),
                "media_share": float(media.median()),
                "control_share": float(control.median()),
            }
        )

prior_r2 = pd.DataFrame(prior_records)
prior_r2.round(3)

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(13, 5.5), sharex=True)
colors = dict(zip(PRIOR_VARIANTS, ["C3", "C1", "C0"], strict=True))

for label in PRIOR_VARIANTS:
    subset = prior_r2.query("prior == @label")
    axes[0].plot(
        subset["K"], subset["r2_median"], marker="o", color=colors[label], label=label
    )
    axes[0].fill_between(
        subset["K"],
        subset["r2_q05"],
        subset["r2_q95"],
        color=colors[label],
        alpha=0.12,
    )

axes[0].axhline(1.0, color="black", linestyle=":", linewidth=1)
axes[0].set(
    xlabel="number of controls K",
    ylabel="implied prior $R^2$",
    title="Implied prior $R^2$ (median, 90% band)",
    ylim=(0, 1.05),
)
axes[0].legend(loc="lower right", fontsize=9)

for label in PRIOR_VARIANTS:
    subset = prior_r2.query("prior == @label")
    axes[1].plot(
        subset["K"],
        subset["media_share"],
        marker="o",
        color=colors[label],
        label=label,
    )

axes[1].set(
    xlabel="number of controls K",
    ylabel="median media share of prior variance",
    title="How much prior variance is left for media",
    ylim=(0, 0.8),
)
axes[1].legend(loc="upper right", fontsize=9)

fig.suptitle(
    "The Normal prior spends its whole variance budget on controls as K grows",
    fontsize=14,
)
fig.tight_layout()

The left panel is the mechanism. Under the library default the implied prior $R^2$ is already indistinguishable from 1 at $K = 5$ — the prior asserts, before seeing any data, that the model explains everything. The calibrated Normal starts in a sensible place and then marches upwards monotonically as $K$ grows, exactly as $K\sigma^2$ predicts. The R2D2 prior is flat: the budget is fixed, so splitting it more finely does not change the total.

The right panel is the same fact told from media's point of view. Under the calibrated Normal, the median share of prior variance available to media collapses as controls are added. Under R2D2 it barely moves. A prior that has decided in advance that media explains almost nothing is a prior that will struggle to attribute revenue to media.

## The experiment

Now the posterior. Nested subsets $K \in$ `K_GRID` of the same pool of controls, so `y` is byte-identical across every fit. Each $K > 0$ is fit twice, once per control prior; $K = 0$ is fit once, since the two priors coincide when there are no controls. Everything else — media priors, seeds, sampler settings — is held identical.

Total ROAS per fit comes from the counterfactual machinery rather than from a contribution deterministic, so it is the same quantity a practitioner would report.

In [ ]:
def total_roas(mmm: MMM) -> np.ndarray:
    """Posterior draws of total media ROAS over the whole fitted period."""
    incremental = mmm.incrementality.compute_incremental_contribution(
        frequency="all_time"
    )
    spend = float(mmm.data.get_channel_spend().sum())
    return (incremental.sum("channel") / spend).values.ravel()


def variance_shares(mmm: MMM, k: int) -> dict:
    """Posterior variance shares and implied posterior R^2 for a fitted model."""
    posterior = mmm.idata.posterior
    var_media = posterior["channel_contribution"].sum("channel").var("date")
    var_control = (
        posterior["control_contribution"].sum("control").var("date")
        if k > 0
        else xr.zeros_like(var_media)
    )
    total = var_media + var_control + posterior["y_sigma"] ** 2

    return {
        "media_share": float((var_media / total).mean()),
        "control_share": float((var_control / total).mean()),
        "posterior_r2": float(((var_media + var_control) / total).mean()),
    }


def fit_for_k(
    dataset: pd.DataFrame, k: int, prior_name: str, seed: int
) -> tuple[np.ndarray, dict]:
    """Fit one (K, prior) cell and return ROAS draws plus fit diagnostics."""
    mmm = build_mmm(k, prior_name)
    mmm.fit(
        dataset.drop(columns=["y"]),
        dataset["y"],
        random_seed=seed,
        progressbar=False,
    )

    diagnostics = az.summary(
        mmm.idata.posterior,
        var_names=["intercept_contribution", "saturation_beta", "y_sigma"],
    )
    info = {
        "K": k,
        "prior": prior_name,
        "divergences": int(mmm.idata.sample_stats["diverging"].sum()),
        "max_r_hat": float(diagnostics["r_hat"].max()),
        "posterior_sigma": float(mmm.idata.posterior["y_sigma"].mean()),
    } | variance_shares(mmm, k)

    return total_roas(mmm), info

The ground-truth check below is worth pausing on. It confirms that our analytic truth is *exactly* the quantity the MMM would report, by plugging the true parameters into the model graph with `pm.do` and reading off its own total media contribution. If the adstock `normalize` flag or the channel scaling were misaligned, this is where it would show up.

In [ ]:
check_mmm = build_mmm(0)
check_mmm.build_model(data.drop(columns=["y"]), data["y"])
fixed = pm.do(
    check_mmm.model,
    {
        "intercept_contribution": truth["intercept"],
        "adstock_alpha": truth["adstock_alpha"],
        "saturation_lam": truth["saturation_lam"],
        "saturation_beta": truth["saturation_beta"],
    },
)
model_total = float(pm.draw(fixed["total_media_contribution_original_scale"]))

print(f"model-implied total ROAS at true parameters: {model_total / spend_matrix.sum():.6f}")
print(f"analytic ground-truth total ROAS          : {truth['total_roas']:.6f}")

In [ ]:
roas_draws: dict[tuple[int, str], np.ndarray] = {}
fit_records = []

for k in K_GRID:
    for prior_name in ["Normal"] if k == 0 else ["Normal", "R2D2"]:
        draws, info = fit_for_k(data, k, prior_name, seed=SEED)
        roas_draws[k, prior_name] = draws
        fit_records.append(info)

fits = pd.DataFrame(fit_records)
fits.round(4)

For $K = 0$ the two priors coincide, so the single $K = 0$ fit is shared by both series. `by_prior` handles that relabelling once, for every table and figure below.

In [ ]:
def roas_summary(draws: np.ndarray) -> dict:
    """Posterior summary of ROAS against the known truth."""
    q05, median, q95 = np.quantile(draws, [0.05, 0.5, 0.95])
    return {
        "q05": q05,
        "median": median,
        "q95": q95,
        "interval_length": q95 - q05,
        "covers_truth": bool(q05 <= truth["total_roas"] <= q95),
        "bias": float(draws.mean() - truth["total_roas"]),
        "squared_error": float((draws.mean() - truth["total_roas"]) ** 2),
    }


def by_prior(frame: pd.DataFrame, prior_name: str) -> pd.DataFrame:
    """One row per K for a single prior, treating the K=0 fit as shared."""
    rows = frame[(frame["prior"] == prior_name) | (frame["K"] == 0)].copy()
    return rows.assign(prior=prior_name).sort_values("K")


roas_table = pd.DataFrame(
    [
        {"K": k, "prior": prior_name} | roas_summary(draws)
        for (k, prior_name), draws in roas_draws.items()
    ]
)
roas_table.round(4)

### The headline figure: total ROAS across $K$

This is the analogue of Figure 9 in the paper. One posterior density per $K$, one column per prior, with the true ROAS and the $K = 0$ baseline marked.

In [ ]:
def series_for(prior_name: str) -> list[tuple[int, np.ndarray]]:
    """ROAS draws by K for one prior, reusing the shared K=0 fit."""
    return [
        (k, roas_draws[k, "Normal" if k == 0 else prior_name]) for k in K_GRID
    ]


baseline = roas_draws[0, "Normal"]
baseline_q05, baseline_q95 = np.quantile(baseline, [0.05, 0.95])

fig, axes = plt.subplots(
    ncols=2, figsize=(13, 6), sharex=True, sharey=True, layout="constrained"
)
palette = sns.color_palette("viridis", n_colors=len(K_GRID))

for ax, prior_name in zip(axes, ["Normal", "R2D2"], strict=True):
    for (k, draws), color in zip(series_for(prior_name), palette, strict=True):
        sns.kdeplot(
            x=draws, ax=ax, color=color, linewidth=2, label=f"K = {k}", clip=(0, None)
        )
    ax.axvline(
        truth["total_roas"], color="black", linestyle="--", linewidth=2, label="truth"
    )
    ax.axvspan(
        baseline_q05,
        baseline_q95,
        color="gray",
        alpha=0.15,
        label="K = 0 baseline, 90%",
    )
    ax.set(xlabel="total ROAS", title=f"{prior_name} prior on controls", xlim=(0, 7))

axes[0].set_ylabel("posterior density")
axes[1].legend(loc="upper right", fontsize=9)

fig.suptitle(
    "Total ROAS posterior as controls are added, by control prior", fontsize=14
)

The two panels tell different stories. Under the R2D2 prior the posterior stays put: adding controls tightens it slightly and leaves it centred near the truth, all the way to $K = 200$. Under the Normal prior the posterior drifts away from the truth and, past $K = 100$, spreads out badly — the control block has become flexible enough to compete with media for the same variation.

A more compact view of the same result: interval length and point-estimate error against $K$.

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(13, 5), sharex=True)
prior_colors = {"Normal": "C1", "R2D2": "C0"}

for prior_name, color in prior_colors.items():
    subset = by_prior(roas_table, prior_name)
    axes[0].plot(
        subset["K"],
        subset["interval_length"],
        marker="o",
        color=color,
        label=f"{prior_name} prior",
    )
    axes[1].plot(
        subset["K"], subset["median"], marker="o", color=color, label=f"{prior_name} prior"
    )
    axes[1].fill_between(
        subset["K"], subset["q05"], subset["q95"], color=color, alpha=0.12
    )

axes[0].set(
    xlabel="number of controls K",
    ylabel="90% interval length",
    title="Posterior interval length for total ROAS",
)
axes[0].legend(fontsize=9)

axes[1].axhline(
    truth["total_roas"], color="black", linestyle="--", linewidth=2, label="truth"
)
axes[1].set(
    xlabel="number of controls K",
    ylabel="total ROAS",
    title="Posterior median and 90% interval",
)
axes[1].legend(fontsize=9)

fig.tight_layout()

### Supporting view: is the variance going to the right place?

Total ROAS is a summary; it helps to see how each fit splits the target's variation between media, controls and residual, and to compare that against the truth. Because the target is max-scaled to a maximum of 1, the model's internal scale is exactly 1, so posterior and true quantities are directly comparable.

In [ ]:
fits[["K", "prior", "media_share", "control_share", "posterior_r2"]].round(4)

In [ ]:
true_media_var_share = truth["r2"] * truth["delta_media"]

fig, axes = plt.subplots(ncols=2, figsize=(13, 5), sharex=True)

for prior_name, color in prior_colors.items():
    subset = by_prior(fits, prior_name)
    axes[0].plot(
        subset["K"],
        subset["media_share"],
        marker="o",
        color=color,
        label=f"{prior_name} prior",
    )
    axes[1].plot(
        subset["K"],
        subset["posterior_r2"],
        marker="o",
        color=color,
        label=f"{prior_name} prior",
    )

axes[0].axhline(
    true_media_var_share,
    color="black",
    linestyle="--",
    linewidth=2,
    label="truth",
)
axes[0].set(
    xlabel="number of controls K",
    ylabel="media share of target variance",
    title="Posterior media variance share",
)
axes[0].legend(fontsize=9)

axes[1].axhline(truth["r2"], color="black", linestyle="--", linewidth=2, label="truth")
axes[1].set(
    xlabel="number of controls K",
    ylabel="posterior $R^2$",
    title=r"Posterior $R^2$: $\mathrm{Var}(\mu) / (\mathrm{Var}(\mu) + \sigma^2)$",
)
axes[1].legend(fontsize=9)

fig.tight_layout()

### Diagnostics

Sampling quality per fit. This matters for interpretation: if a fit that produces a badly placed ROAS posterior also shows no sampling trouble, then nothing would have warned a practitioner that something was wrong.

In [ ]:
fits.assign(
    r_hat_flag=lambda frame: np.where(frame["max_r_hat"] > 1.01, "check", ""),
).round(4)

### Repetition study

A single dataset cannot separate a real effect from one draw's luck, and the paper's own effect is modest — Table 4 reports RMSE 0.30 against 0.27 and coverage 0.87 against 0.92 over many replications. The function below is the Table 4 analogue: repeat the whole pipeline over `n_reps` datasets and report 90% interval length, coverage and RMSE of total ROAS per $K$ and prior.

It defaults to `n_reps=1`, which reuses the dataset above so the notebook stays fast; raising `n_reps` is the only change needed to turn this into a proper Monte Carlo study. With `n_reps=1`, "coverage" is a single indicator per cell rather than a rate, and RMSE is the absolute error of one posterior mean — read the table as bookkeeping, and the figures above as the evidence.

In [ ]:
def run_experiment(n_reps: int = 1, base_seed: int = SEED) -> pd.DataFrame:
    """Repeat simulate + fit over `n_reps` datasets."""
    records = []
    for rep in range(n_reps):
        seed = base_seed + rep
        if rep == 0:
            rep_data, rep_truth = data, truth
        else:
            rep_data, rep_truth = simulate(seed)

        for k in K_GRID:
            for prior_name in ["Normal"] if k == 0 else ["Normal", "R2D2"]:
                if rep == 0:
                    draws = roas_draws[k, prior_name]
                else:
                    draws, _ = fit_for_k(rep_data, k, prior_name, seed=seed)

                q05, q95 = np.quantile(draws, [0.05, 0.95])
                records.append(
                    {
                        "rep": rep,
                        "K": k,
                        "prior": prior_name,
                        "interval_length": q95 - q05,
                        "covers": bool(q05 <= rep_truth["total_roas"] <= q95),
                        "squared_error": (draws.mean() - rep_truth["total_roas"]) ** 2,
                    }
                )
    return pd.DataFrame(records)


reps = run_experiment(n_reps=1)

reps.groupby(["K", "prior"]).agg(
    interval_length=("interval_length", "mean"),
    coverage=("covers", "mean"),
    rmse=("squared_error", lambda errors: np.sqrt(errors.mean())),
).round(4)

## Conclusion

Adding a control variable to an MMM is not free, and the cost is not paid in the place you would look for it. The controls here are neutral by construction — independent of spend, incapable of confounding ROAS — and the data never change across fits. Everything we measured came from the prior on `gamma_control`.

What the two figures show:

- **The mechanism is visible before any MCMC.** Under an independent Normal prior with a fixed `sigma`, the implied prior $R^2$ rises monotonically towards 1 as controls are added, and the share of prior variance available to media collapses. Under the library default `sigma=2`, this has already happened at $K = 5$. The R2D2 prior is flat in $K$, because the Dirichlet splits a fixed budget rather than accumulating one.
- **It reaches the reported number.** Under the Normal prior the total-ROAS posterior drifts off the truth and, once $K$ is comparable to $n$, widens sharply. Under the split R2D2 prior it stays centred and its interval keeps tightening as genuinely informative controls are added.

Practical takeaways:

1. **The default `gamma_control` prior does not know how many controls you have.** `Prior("Normal", mu=0, sigma=2, dims="control")` is a strong statement about total explained variance once $K$ is more than a handful, and it gets stronger every time you add a column.
2. **Check the implied prior $R^2$.** It costs one `sample_prior_predictive` call and no sampling, and it is the fastest way to find out whether your control prior is quietly asserting a near-perfect fit.
3. **Standardise your controls.** Both priors are stated on the scale of the coefficients, and `MMM` does not scale controls for you. A shared variance budget is only meaningful once the columns are comparable.
4. **A split prior is enough.** Putting R2D2 on the controls alone leaves every media prior untouched, so the media parameterisation you have tuned stays exactly as it was. The joint version, which shrinks media too, is a different and — per the paper — considerably worse idea.
5. **Diagnostics will not save you.** Look at the divergence and $\hat{R}$ columns: for most of the grid the Normal fits look perfectly healthy while their ROAS posterior is drifting. Only in the most extreme cell does sampling itself complain.

### What this notebook does not do

- **The joint R2D2 prior**, spanning media and control coefficients together, which the paper shows is badly biased.
- **Multiple geos.** The class used here is the dims-based `MMM` either way, so adding `dims=("geo",)` is mostly a matter of widening the prior dims and the spend generator.
- **Promoting `R2D2Prior` into `pymc_marketing.special_priors`** with tests and serialisation registration. It is defined locally here; upstreaming it alongside `LaplacePrior` and `LogNormalPrior` would be a natural follow-up.
- **Conditioning the R2D2 budget on the sampled residual `sigma`**, which the current `gamma_control` hook cannot reach.

## References

- Zhang, Y. D., Naughton, B. P., Bondell, H. D., & Reich, B. J. (2022). [Bayesian Regression Using a Prior on the Model Fit: The R2-D2 Shrinkage Prior](https://doi.org/10.1080/01621459.2020.1825449). *Journal of the American Statistical Association*.
- Cinelli, C., Forney, A., & Pearl, J. (2024). [A Crash Course in Good and Bad Controls](https://doi.org/10.1177/00491241221099552). *Sociological Methods & Research*.
- Jin, Y., Wang, Y., Sun, Y., Chan, D., & Koehler, J. (2017). [Bayesian Methods for Media Mix Modeling with Carryover and Shape Effects](https://research.google/pubs/pub46001/).

In [ ]:
%load_ext watermark
%watermark -n -u -v -iv -w -p pymc_marketing,pytensor,pymc